**2 Exploracion de datos**

In [ ]:
# importamos la función cargarDatos del módulo carga_datos para cargar los datos necesarios para el programa
from carga_datos import cargarDatos

In [ ]:
# cargamos los datos utilizando el método 'cargarDatos' hecho en el script 'cargar_datos.py'

df = cargarDatos()

In [ ]:
df.info()

In [ ]:
df.head()

**2.1 Revision de nulos**

In [ ]:
#####
df.isnull().mean().sort_values(ascending=False)

Se eliminarán aquellos atributos que presenten un nivel muy alto de valores nulos, ya que requerirían un esfuerzo considerable para imputarlos y podrían introducir sesgos en el análisis. En el caso de promedio_ingresos_datacredito y tendencia_ingresos, ambas presentan alrededor del 30% de valores faltantes. Si bien este porcentaje podría justificar su eliminación, se ha decidido conservarlas temporalmente con el fin de evaluarlas en etapas posteriores y determinar si aportan valor predictivo o deben finalmente descartarse.

**2.2 Unificar la forma como se representan los valores Nulos.**

In [ ]:
#!pip install numpy
import numpy as np

In [ ]:
df.isin(["NA", "N/A", "null", "NULL", "?", "Sin dato"]).sum()

In [ ]:
import numpy as np
df = df.replace(["NA", "N/A", "null", "NULL", "?", "Sin dato"], np.nan)

**2.3 Eliminacion variables irrelevantes**

In [ ]:
df = df.drop(columns=["fecha_prestamo","plazo_meses"])

No era relevante para los objetivos iniciales de exploración, con cuota pactada y capital prestado se puede saber el plazo de pago.

**2.4 Conversión del tipo de datos**

Se convierten las variables categoricas tipo object a category

In [ ]:
df["tipo_credito"] = df["tipo_credito"].astype("category")
df["tipo_laboral"] = df["tipo_laboral"].astype("category")
df["tendencia_ingresos"] = df["tendencia_ingresos"].astype("category")

In [ ]:
df.info()

**3 EDA**

**3.1 Analisis Univariable**


Para las variables numéricas realizamos un describe

In [ ]:
#Variables numericas
df.describe()

In [ ]:
df_num = df.select_dtypes(include=["int64", "float64"])
df_num.agg(["mean", "median", "std", "var", "min", "max", "skew", "kurt"])

Para las variables categoricas un value_counts()

In [ ]:
df.tipo_laboral.value_counts()

In [ ]:
df.tipo_credito.value_counts()

In [ ]:
df.tendencia_ingresos.value_counts()

Podemos ver que hay un problema de calidad de datos en la variable tendencia_ingresos por lo que tenemos que corregir esta variable dejando las categorias esperadas (Creciente, Decreciente, Estable) y eliminar los datos numericos que no corresponden a una variable categorica

In [ ]:
categorias_validas = ["Creciente", "Decreciente", "Estable"]
df["tendencia_ingresos"] = df["tendencia_ingresos"].where(
    df["tendencia_ingresos"].isin(categorias_validas),
    np.nan
)

df["tendencia_ingresos"] = df["tendencia_ingresos"].astype("category")
df["tendencia_ingresos"] = df["tendencia_ingresos"].cat.remove_unused_categories()

La variable tendencia_ingresos presentaba valores erróneos (numéricos que no corresponden a su naturaleza categórica). Estos registros representaban menos del 1% del total, por lo cual se reemplazaron por NaN. El total de categorías válidas pasó de 7,831 a 7,773 registros, manteniendo la coherencia de la variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filtrar solo columnas numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Recorrer y graficar histogramas
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col].dropna(), bins=30)
    plt.title(f"Histograma de {col}", fontsize=14)
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.show()

**Capital Prestado:** Distribución sesgada hacia la derecha con la mayoría de préstamos concentrados en valores bajos. Presencia de outliers sugiere diferentes segmentos de clientes.

**Plazo Meses:** Distribución relativamente uniforme con concentración en plazos estándar (12, 24, 36 meses). En términos de cumplimiento, los créditos de corto plazo son más confiables, y el riesgo aumenta en mediano y largo plazo.

**Edad Cliente:** Distribución normal con ligero sesgo hacia edades mayores. Concentración principal entre 30-50 años, típico de población crediticiamente activa.

**Salario Cliente** Fuerte concentración en salarios bajos con cola larga hacia valores altos. Patrón típico de distribución de ingresos en población general.

**Total Otros Préstamos:** Mayoría de clientes sin otros préstamos, con casos aislados de exposiciones múltiples muy altas. Sugiere portfolio principalmente de primeros créditos.

**Cuota Pactada:** Distribución sesgada hacia la derecha con concentración en cuotas bajas. Distribución similar a capital, con mayoría en valores bajos y pocos extremos altos.

**Puntaje:** Altamente concentrado en puntajes entre 80-100.

**Puntaje Datacrédito**: Similar al puntaje anterior con mayor concentración en 800.

**Cantidad Créditos Vigentes:** Fuerte concentración entre 0 - 10. Pocos clientes con múltiples exposiciones, indicando gestión conservadora de riesgo.

**Huella Consulta:** Distribucion sesgada hacia la derecha alta concentracion de personas entre 0 - 5.

**Saldo Mora:** Mayoría de clientes sin mora (valores en cero), con casos aislados de moras significativas.

**Saldo Total:** Gran variabilidad con mediana muy inferior a la media, confirmando concentración en saldos bajos con outliers extremos.

**Saldo Principal:** Patrón similar al saldo total.

**Saldo Mora Codeudor:** Mayoría absoluta en cero, indicando baja utilización de codeudores y/o buen comportamiento de pago conjunto.

**Créditos Sector Financiero:** Distribución concentrada en valores bajos, con mayoría de clientes teniendo pocos o ningún crédito en sector financiero.

**Créditos Sector Cooperativo:** Similar al sector financiero, concentración en valores bajos sugiriendo baja penetración del sector cooperativo.

**Estadísticas Créditos Sector Real:** Baja exposición promedio en sector real, consistente con el perfil de cartera observado.

**Promedio Ingresos Datacrédito:** Distribución sesgada similar al salario cliente.

In [ ]:
# Filtrar solo columnas numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

# Recorrer y graficar boxplots
for col in num_cols:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col].dropna())
    plt.title(f"Boxplot de {col}", fontsize=14)
    plt.xlabel(col)
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.show()

**Boxplot Capital Prestado**
* Mediana muy baja (~500K) con Q3 alrededor de 2M, indicando que 75% de los préstamos son de monto pequeño-mediano. Outliers extremos hasta 50M+ sugieren préstamos corporativos o hipotecarios. La caja compacta confirma concentración en segmento retail.

**Boxplot Edad Cliente**
* Distribución simétrica con mediana ~40 años. Rango intercuartílico normal (30-50 años) sin outliers extremos, confirmando población crediticia estándar. Los whiskers se extienden apropiadamente hasta los límites naturales de edad crediticia.

**Boxplot Salario Cliente**
* Mediana baja (~2M) con Q3 alrededor de 4M, mostrando concentración en ingresos medios-bajos. Outliers extremos hasta 50M+ indican ejecutivos o empresarios. La asimetría hacia arriba es típica de distribuciones de ingreso.

**Boxplot Total Otros Préstamos**
* Mediana en cero confirma que mayoría de clientes no tienen otros préstamos. Q3 bajo indica que incluso el 75% superior tiene exposiciones limitadas. Outliers significativos sugieren clientes con múltiples obligaciones financieras.

**Boxplot Cuota Pactada**
* Mediana baja (~200K) consistente con ingresos y montos prestados observados. La proporción cuota/ingreso implícita parece conservadora. Outliers extremos corresponden a los préstamos de mayor monto identificados previamente.

**Boxplot Cantidad Créditos Vigentes**
* Mediana en 1-2 créditos con Q3 alrededor de 3, indicando gestión conservadora de exposición múltiple. Pocos outliers con muchos créditos sugieren casos especiales o clientes premium con mayor capacidad de endeudamiento.

**Boxplot Saldos (Variables múltiples)**
* Todos muestran patrón similar: medianas bajas, alta concentración en valores pequeños y outliers extremos. Confirma que el portfolio está dominado por créditos pequeños con casos aislados de exposiciones significativas.

In [ ]:
# Filtrar numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop("Pago_atiempo")

# Generar tablas pivote
for col in num_cols:
    tabla = df.pivot_table(values=col, index="Pago_atiempo", aggfunc=["mean", "median", "min", "max"])
    print(f"\n📊 Tabla pivote para {col} con respecto a Pago_atiempo:")
    print(tabla)

In [ ]:
# Buscar columnas categóricas directamente
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns



for col in categorical_cols:
    plt.figure(figsize=(10, 6))
    df[col].value_counts().plot(kind="bar")
    plt.title(f"Frecuencia de {col}")
    plt.xlabel(col)
    plt.ylabel("Conteo")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    print(f"Value counts for {col}:\n{df[col].value_counts()}\n")
    print("-" * 50)

**Tendencia ingresos**
Fuerte desbalance con una categoría dominante representando ~80% de los casos.

**Tipo credito**

* Existe una categoría claramente dominante (probablemente consumo o libre inversión).

* Las demás categorías aparecen en proporciones mucho menores.

* Esto podría generar desbalance en las clases y requerir técnicas de balanceo o agrupación.

**Tipo laboral**

* Mayoría de clientes son empleados, mientras que los independientes representan un grupo menor.

* Podría ser relevante porque los independientes suelen tener mayor probabilidad de incumplimiento.

**Analisis bivariable**

Nos interesa revisar cada variable respecto a la objetivo. Para las variables numéricas boxplot utilizando como hue o columna la variable objetivo. Hisplot con hue=columna objetivo

In [ ]:
# Variables numéricas
num_cols = df.select_dtypes(include=["int64", "float64"]).drop(columns=["Pago_atiempo"]).columns
for col in num_cols:
    plt.figure(figsize=(7,5))
    sns.histplot(data=df, x=col, hue="Pago_atiempo", element="step", stat="density", common_norm=False)
    plt.title(f"Distribución de {col} según Pago_atiempo", fontsize=14)
    plt.xlabel(col)
    plt.ylabel("Densidad")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.show()

**Capital Prestado vs Pago a Tiempo:**
* Los clientes que no pagan a tiempo tienden a tener préstamos de montos más altos. La distribución muestra mayor dispersión en el grupo de no pagadores, sugiriendo que montos elevados aumentan el riesgo de incumplimiento.

**Plazo Meses vs Pago a Tiempo:**
* Los buenos pagadores se concentran en plazos medios (12-36 meses), mientras que los malos pagadores muestran mayor variabilidad. Plazos muy largos parecen asociarse con mayor riesgo de incumplimiento.

**Edad Cliente vs Pago a Tiempo:**
* Diferencia notable: clientes que pagan a tiempo tienden a ser mayores (concentración 35-50 años). Los más jóvenes muestran mayor propensión al incumplimiento, confirmando edad como factor de riesgo.

**Salario Cliente vs Pago a Tiempo:**
* Los buenos pagadores muestran distribución más concentrada en ingresos medios-altos. Los malos pagadores tienen mayor dispersión, incluyendo casos de ingresos altos que no pagan, sugiriendo otros factores de riesgo.

**Total Otros Préstamos vs Pago a Tiempo:**
* Clara diferenciación: malos pagadores tienen significativamente más exposición en otros préstamos. Alta correlación entre sobreendeudamiento y riesgo de incumplimiento.

**Cuota Pactada vs Pago a Tiempo:**
* Los buenos pagadores se concentran en cuotas bajas-medias. Cuotas muy altas se asocian con mayor riesgo, probablemente por compromiso excesivo de la capacidad de pago.

**Puntaje vs Pago a Tiempo:**
* Diferenciación muy clara: buenos pagadores tienen scores superiores (600-800), malos pagadores concentrados en scores bajos (300-500). Confirma el poder predictivo del scoring crediticio.

**Puntaje Datacrédito vs Pago a Tiempo:**
* Similar al puntaje anterior pero con diferenciación aún más marcada. Los buenos pagadores están claramente en rangos altos (700+), validando la calidad de esta fuente de información.

**Cantidad Créditos Vigentes vs Pago a Tiempo:**
* Los buenos pagadores tienden a tener 1-2 créditos, mientras que los malos pagadores muestran mayor dispersión incluyendo casos con muchos créditos vigentes. Sobreendeudamiento como indicador de riesgo.

**Huella Consulta vs Pago a Tiempo:**
* Los malos pagadores tienen más consultas crediticias, sugiriendo búsqueda desesperada de crédito. Los buenos pagadores muestran comportamiento más estable con pocas consultas.

In [ ]:
# Filtrar columnas categóricas
cat_cols = df.select_dtypes(include=['object', 'category']).columns

# Recorrer y graficar
for col in cat_cols:
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=col, hue="Pago_atiempo")
    plt.title(f"Distribución de {col} según Pago_atiempo", fontsize=14)
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.legend(title="Pago a tiempo", labels=["No", "Sí"])
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.show()

**Tipo 4 y 9:**

* 4.7% de incumplimiento en ambos tipos

* 95.3% pagan a tiempo - Performance sólida

* Productos crediticios de riesgo controlado

**Tipo 6 (Riesgo Alto):**

* 42.9% de incumplimiento - Significativamente mayor

* 57.1% pagan a tiempo - Performance deficiente


**Tipo 10**

* 2.6% de incumplimiento - Excelente resultado
* 97.4% pagan a tiempo


In [ ]:
plt.figure(figsize=(12,8))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Matriz de correlación entre variables numéricas", fontsize=16)
plt.show()

**Correlaciones más fuertes (≥ 0.7):**

*Correlaciones positivas muy altas*
* Pago_atiempo con puntaje (0.92) - Los clientes que pagan a tiempo tienen mejores puntajes crediticios

* creditos_sectorFinanciero con cant_creditosvigentes (0.79) - Más créditos en sector financiero se asocia con más créditos vigentes

* saldo_total con saldo_principal (0.73) - Relación directa entre saldos totales y principales

**Correlaciones moderadas (0.4-0.7):**

* salario_cliente con total_otros_prestamos (0.60) - Salarios más altos se asocian con más préstamos

* creditos_sectorReal con cant_creditosvigentes (0.54) - Créditos en sector real correlacionan con cantidad de créditos vigentes

* saldo_principal con creditos_sectorFinanciero (0.54)

**Correlaciones negativas notables:**

* edad_cliente con puntaje_datacredito (-0.38) - Los clientes mayores tienden a tener puntajes más bajos

* cuota_pactada con puntaje_datacredito (-0.28) - Cuotas más altas se asocian con puntajes más bajos

* huellas_consulta con edad_cliente (-0.24) - Los clientes mayores tienen menos consultas de crédito


In [ ]:
sns.pairplot(df, vars=["capital_prestado", "salario_cliente", "puntaje", "edad_cliente"],
             hue="Pago_atiempo", diag_kind="kde", corner=True)
plt.suptitle("Pairplot de variables numéricas con respecto a Pago_atiempo", y=1.02, fontsize=14)
plt.show()


## Capital Prestado vs Salario Cliente
**Separación clara**: Los buenos pagadores (naranja) se concentran en montos bajos-medios con salarios proporcionales. Los malos pagadores (azul) muestran mayor dispersión, incluyendo casos de préstamos altos con salarios bajos (riesgo evidente).

## Capital Prestado vs Puntaje
**Discriminación fuerte**: Buenos pagadores tienen puntajes consistentemente altos (>600) independientemente del monto. Malos pagadores concentrados en puntajes bajos (<500), especialmente con montos altos.

## Capital Prestado vs Edad
**Patrón de riesgo por edad**: Clientes jóvenes (<30) con préstamos altos predominantemente azules (mayor riesgo). Buenos pagadores se distribuyen más uniformemente en edad pero con concentración en 35-55 años.

## Salario vs Puntaje
**Correlación de calidad crediticia**: Cluster denso de buenos pagadores en la zona de salarios medios-altos + puntajes altos. Malos pagadores dispersos, algunos con salarios altos pero puntajes bajos (red flag).

## Salario vs Edad
**Estabilidad por madurez**: Buenos pagadores muestran correlación positiva salario-edad (progresión natural de carrera). Malos pagadores más erráticos en esta relación.

## Puntaje vs Edad
**Separación más clara del pairplot**: Línea de separación casi perfecta. Buenos pagadores dominan rangos de puntaje alto, concentrados en edades maduras. La edad temprana + puntaje bajo = riesgo muy alto.

In [ ]:
pd.crosstab([df["tipo_laboral"], df["tipo_credito"]], df["Pago_atiempo"], normalize="index") * 100



### Patrones por Tipo de Crédito:

#### **Tipo 4 (Empleados vs Independientes):**
- **Empleados**: 4.14% incumplimiento vs **Independientes**: 5.71%
- Los independientes tienen **38% más riesgo** en este producto
- Diferencia moderada pero consistente con el patrón general

#### **Tipo 6 (Alto Riesgo - Solo Empleados):**
- **42.86% de incumplimiento** - Confirma que es el producto más riesgoso
- No hay independientes en este tipo de crédito (posible restricción de política)

#### **Tipo 9:**
- **Empleados**: 4.36% vs **Independientes**: 5.19%
- Diferencia pequeña, ambos con buen desempeño

#### **Tipo 10:**
- **Empleados**: 2.67% vs **Independientes**: 2.44%
- **Sorpresa**: Independientes ligeramente mejor que empleados
- Posible producto especializado para independientes

#### **Consistencia del Patrón:**
En la mayoría de productos, los **independientes tienen mayor riesgo**, excepto en el Tipo 10 donde la diferencia es mínima.

#### **Implicaciones de Pricing:**
Los independientes podrían justificar **tasas ligeramente superiores** en la mayoría de productos, excepto Tipo 10.


In [ ]:
pd.crosstab([df["tendencia_ingresos"],df["tipo_laboral"]], df["Pago_atiempo"], normalize="index") * 100


### Patrones por Tendencia de Ingresos:

#### **Ingresos Crecientes (Mejor Performance):**
- **Empleados**: 3.56% incumplimiento - **Excelente**
- **Independientes**: 4.82% incumplimiento - **Bueno**
- **Gap**: Independientes 35% más riesgo, pero ambos en rangos buenos

#### **Ingresos Decrecientes (Mayor Riesgo):**
- **Empleados**: 5.87% incumplimiento - **Regular**
- **Independientes**: 7.34% incumplimiento - **Alto riesgo**
- **Gap**: Independientes 25% más riesgo, ambos preocupantes

#### **Ingresos Estables (Performance Intermedia):**
- **Empleados**: 4.89% incumplimiento - **Aceptable**
- **Independientes**: 4.80% incumplimiento - **Aceptable**
- **Sorpresa**: Independientes ligeramente mejor que empleados


#### **Jerarquía de Riesgo:**
1. **Menor riesgo**: Ingresos crecientes + Empleado (3.56%)
2. **Riesgo bajo**: Ingresos crecientes + Independiente (4.82%)
3. **Riesgo medio**: Ingresos estables (ambos ~4.8%)
4. **Riesgo alto**: Ingresos decrecientes + Empleado (5.87%)
5. **Mayor riesgo**: Ingresos decrecientes + Independiente (7.34%)

#### **Patrones Contraintuitivos:**
- Con ingresos **estables**, los independientes son ligeramente **menos riesgosos** que empleados
- La **tendencia de ingresos es más predictiva** que el tipo laboral


#### **Para Políticas:**
- **Rechazar/encarecer**: Independientes con ingresos decrecientes
- **Promover**: Cualquier perfil con ingresos crecientes
- **Reevaluar**: Ingresos estables requieren análisis adicional

